In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras

keras.utils.set_random_seed(42)
epochs = 10


In [ ]:
train_ds, val_ds, test_ds = tfds.load(
    "tf_flowers",
    split=["train[:70%]", "train[70%:85%]", "train[85%:]"],
    as_supervised=True,
)


In [ ]:
def prepare(dataset, training=False):
    dataset = dataset.map(
        lambda image, label: (
            keras.applications.vgg16.preprocess_input(tf.image.resize(image, (150, 150))), label
        ),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    if training:
        dataset = dataset.shuffle(256, seed=42)
    return dataset.batch(32).prefetch(tf.data.AUTOTUNE)

train_ds = prepare(train_ds, training=True)
val_ds = prepare(val_ds)
test_ds = prepare(test_ds)


In [ ]:
base_model = keras.applications.VGG16(
    weights="imagenet", include_top=False, input_shape=(150, 150, 3),
)
base_model.trainable = False


In [ ]:
classifier = keras.Sequential([
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(50, activation="relu"),
    keras.layers.Dense(20, activation="relu"),
    keras.layers.Dense(5, activation="softmax"),
])
model = keras.Sequential([base_model, classifier])


In [ ]:
model.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"],
)
model.fit(
    train_ds, validation_data=val_ds, epochs=epochs,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
)


In [ ]:
loss, accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test loss: {loss:.4f}, test accuracy: {accuracy:.4f}")
